**Part 3 · LangGraph** · step 16 of 25  
[← 15 Custom middleware](../02_LangChain/15_custom_middleware.ipynb) · [🏠 Roadmap](../README.md) · [17 State & reducers →](17_state_and_reducers.ipynb)

# 16 · Your First LangGraph — State, Nodes, Edges

## Why LangGraph?
`create_agent` (notebook 10) gives you one fixed loop: *model → tools → model*.
Real applications often need more control:
- **custom steps & branching** — classify first, then route to different handlers;
- **loops with rules** — retry, reflect, stop after N steps;
- **several agents / roles** working together;
- **persistence & pause/resume** — long-running jobs, human approval mid-way.

**LangGraph** lets you draw that flow explicitly as a **graph**: nodes are Python functions,
edges decide what runs next, and a shared **state** carries data between them.
(`create_agent` itself is a LangGraph graph — so everything here also explains what it does internally.)

| Piece | What it is |
|---|---|
| **State** | A `TypedDict` shared by all nodes — the graph's memory while it runs |
| **Node** | A plain function `state -> dict` that returns **only the keys it updates** |
| **Edge** | A fixed transition `A → B` |
| **Conditional edge** | A router function that looks at the state and **returns the name** of the next node |
| `START` / `END` | Special entry and exit points |
| `compile()` | Validates the graph and makes it runnable (`invoke`, `stream`, …) |

Graph we build:
```
START → start_play → (random) → cricket   → END
                             ↘ badminton → END
```
*Runs offline — no API key needed.*

In [ ]:
from typing_extensions import TypedDict


class State(TypedDict):
    """Graph state: a single string each node appends to."""

    graph_info: str

In [ ]:
def start_play(state: State) -> dict:
    """Entry node: start the sentence."""
    print("start_play node has been called")
    return {"graph_info": state["graph_info"] + " I am planning to play"}


def cricket(state: State) -> dict:
    """One branch of the conditional edge."""
    print("cricket node has been called")
    return {"graph_info": state["graph_info"] + " Cricket"}


def badminton(state: State) -> dict:
    """The other branch of the conditional edge."""
    print("badminton node has been called")
    return {"graph_info": state["graph_info"] + " Badminton"}

## Router for the conditional edge
The return type `Literal[...]` lists the possible next nodes — LangGraph uses it to draw the graph.

In [ ]:
import random
from typing import Literal


def random_play(state: State) -> Literal["cricket", "badminton"]:
    """Pick the next node at random (a real router would inspect the state)."""
    return "cricket" if random.random() > 0.5 else "badminton"

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)

# 1. nodes
builder.add_node("start_play", start_play)
builder.add_node("cricket", cricket)
builder.add_node("badminton", badminton)

# 2. edges
builder.add_edge(START, "start_play")
builder.add_conditional_edges("start_play", random_play)
builder.add_edge("cricket", END)
builder.add_edge("badminton", END)

# 3. compile
graph = builder.compile()

In [ ]:
from IPython.display import Image, display


def show_graph(compiled_graph):
    """Render a compiled graph as a PNG.

    draw_mermaid_png() uses the mermaid.ink web service; if you are offline
    it falls back to printing the Mermaid source (paste it into mermaid.live).
    """
    try:
        display(Image(compiled_graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(compiled_graph.get_graph().draw_mermaid())


    show_graph(graph)

In [ ]:
graph.invoke({"graph_info": "My name is Ujjawal."})

## Revision notes
- Build order: **State → nodes → edges → `compile()` → `invoke()`**.
- A node returns a *partial* update; keys it doesn't return are left unchanged.
- Without a reducer, a returned key **overwrites** the old value (next notebook explains reducers).

---
✅ **Done with 16 First graph.** Next: [17 State & reducers →](17_state_and_reducers.ipynb)  
[🏠 Roadmap](../README.md)